In [19]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from pytorchvideo.models.hub import x3d_m
from ultralytics import YOLO

VIDEO_PATH = "/content/303128_tiny.jpg"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_FRAMES = 16
IMAGE_SIZE = 224
THRESHOLD = 0.4
DETECTOR_WEIGHTS = "yolo26n.pt"
PERSON_CONF = 0.4
MIN_GROUP_SIZE = 3
GROUP_DIST_FACTOR = 1.0
CROP_PADDING = 0.15


def load_violence_model():
    ckpt_path = hf_hub_download(
        repo_id="visionlab-ai/school-violence-detection-models",
        filename="final/final_x3d_realtime.pt",
    )
    model = x3d_m(pretrained=False)
    model.blocks[5].proj = nn.Sequential(nn.Dropout(0.5), nn.Linear(2048, 2))

    np_core = getattr(np, "_core", None) or np.core
    safe = [np_core.multiarray.scalar, np.dtype]
    if hasattr(np, "dtypes"):
        safe += [getattr(np.dtypes, n) for n in dir(np.dtypes) if n.endswith("DType")]
    with torch.serialization.safe_globals(safe):
        checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=True)

    if "model" in checkpoint:
        state_dict = checkpoint["model"]
    elif "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    state_dict = {
        (k[len("backbone."):] if k.startswith("backbone.") else k): v
        for k, v in state_dict.items()
    }
    model.load_state_dict(state_dict)
    return model.to(DEVICE).eval()


def read_frames(path, num_frames):
    if path.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
        img = cv2.imread(path)
        if img is None:
            raise RuntimeError(f"Could not read image: {path}")
        return [img.copy() for _ in range(num_frames)]

    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {path}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise RuntimeError("Video contains no frames.")

    indices = torch.linspace(0, total - 1, num_frames).long().tolist()
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    cap.release()

    if not frames:
        raise RuntimeError("Could not read frames.")
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())
    return frames[:num_frames]


def to_clip_tensor(frames_bgr):
    out = []
    for f in frames_bgr:
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        f = cv2.resize(f, (IMAGE_SIZE, IMAGE_SIZE))
        f = torch.from_numpy(f).permute(2, 0, 1).float() / 255.0
        out.append(f)
    video = (torch.stack(out) - 0.45) / 0.225
    return video.permute(1, 0, 2, 3).unsqueeze(0)


@torch.no_grad()
def violence_prob(model, frames_bgr):
    video = to_clip_tensor(frames_bgr).to(DEVICE)
    return F.softmax(model(video), dim=1)[0][1].item()


def find_groups(boxes):
    n = len(boxes)
    if n == 0:
        return []
    centres = np.stack([(boxes[:, 0] + boxes[:, 2]) / 2,
                        (boxes[:, 1] + boxes[:, 3]) / 2], axis=1)
    heights = boxes[:, 3] - boxes[:, 1]
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i in range(n):
        for j in range(i + 1, n):
            if np.linalg.norm(centres[i] - centres[j]) < GROUP_DIST_FACTOR * max(heights[i], heights[j]):
                parent[find(i)] = find(j)

    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())


def detect_groups(detector, frames_bgr):
    results = detector(frames_bgr, classes=[0], conf=PERSON_CONF, verbose=False)
    counts, sizes = [], []
    union = None
    for res in results:
        boxes = res.boxes.xyxy.cpu().numpy() if len(res.boxes) else np.zeros((0, 4))
        groups = [g for g in find_groups(boxes) if len(g) >= MIN_GROUP_SIZE]
        counts.append(len(groups))
        if not groups:
            sizes.append(0)
            continue
        best = max(groups, key=len)
        sizes.append(len(best))
        gb = boxes[best]
        x1, y1, x2, y2 = gb[:, 0].min(), gb[:, 1].min(), gb[:, 2].max(), gb[:, 3].max()
        if union is None:
            union = [x1, y1, x2, y2]
        else:
            union = [min(union[0], x1), min(union[1], y1), max(union[2], x2), max(union[3], y2)]
    return int(np.median(counts)), int(np.median(sizes)), union


def crop_clip(frames_bgr, box):
    h, w = frames_bgr[0].shape[:2]
    x1, y1, x2, y2 = box
    pad_x, pad_y = (x2 - x1) * CROP_PADDING, (y2 - y1) * CROP_PADDING
    x1, y1 = int(max(0, x1 - pad_x)), int(max(0, y1 - pad_y))
    x2, y2 = int(min(w, x2 + pad_x)), int(min(h, y2 + pad_y))
    if x2 - x1 < 32 or y2 - y1 < 32:
        return None
    return [f[y1:y2, x1:x2] for f in frames_bgr]


def main():
    violence_model = load_violence_model()
    detector = YOLO(DETECTOR_WEIGHTS)
    frames = read_frames(VIDEO_PATH, NUM_FRAMES)

    prob = violence_prob(violence_model, frames)
    num_groups, largest, crop_box = detect_groups(detector, frames)

    if crop_box is not None:
        cropped = crop_clip(frames, crop_box)
        if cropped is not None:
            prob = max(prob, violence_prob(violence_model, cropped))

    print(f"Number of groups: {num_groups}")
    print(f"People in largest group: {largest}")
    print(f"Violence: {'Yes' if prob > THRESHOLD else 'No'}")


if __name__ == "__main__":
    main()

Number of groups: 1
People in largest group: 12
Violence: No
